# 1. What you'll learn

        - turn a series into input/target windows
- inspect LSTM hidden tensor shapes
- compare one-step fit with recursive long-horizon failure

        **The one question this notebook answers:** How does an LSTM carry selected information through time, and why do long recursive forecasts drift?

# 2. Setup

This lesson keeps rnns and lstms for sequences small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A noisy signal combines two periodic frequencies and a slowly changing amplitude. Twenty-four past values predict the next value; the last 200 windows are held out chronologically.

**Small example:** For five values [1,2,3,4,5] and window length 3, training pairs are [1,2,3]→4 and [2,3,4]→5; a recursive two-step forecast must feed its predicted 4 back to predict 5.

In [ ]:
rng=np.random.default_rng(SEED);t=np.arange(1400);values=(1+.0005*t)*np.sin(2*np.pi*t/40)+.35*np.sin(2*np.pi*t/9)+rng.normal(0,.08,len(t));L=24;X=np.array([values[i:i+L] for i in range(len(values)-L)]);y=np.array([values[i+L] for i in range(len(values)-L)]);print("Series:",values.shape,"windows/targets:",X.shape,y.shape)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(t[:240],values[:240]);axes[0].set_title("Two-frequency sequence");axes[1].hist(values,bins=30,color="#176b66");axes[1].set_title("Value distribution")
axes[2].plot(range(L),X[10],marker=".",label="input");axes[2].scatter([L],[y[10]],color="#d38b45",label="target");axes[2].set_title("One supervised window");axes[2].legend();plt.tight_layout();plt.show()

**Takeaway:** The signal has short and long periodic memory.

**Takeaway:** Values are centred but later amplitude grows slowly.

**Takeaway:** Each target is strictly after all values in its input window.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
cut=len(X)-200;mean=X[:cut].mean();std=X[:cut].std();Xa=((X[:cut]-mean)/std)[:,:,None];ya=((y[:cut]-mean)/std)[:,None];Xb=((X[cut:]-mean)/std)[:,:,None];yb=y[cut:];loader=DataLoader(TensorDataset(torch.tensor(Xa,dtype=torch.float32),torch.tensor(ya,dtype=torch.float32)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));print("Train/test tensors:",Xa.shape,Xb.shape)

# 6. Train

        Training turns the assumptions behind rnns and lstms for sequences into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Read one scalar and previous state at each time step.
2. LSTM gates decide what to forget, write, and expose.
3. Use the final hidden state to predict the next value.
4. Backpropagate loss through all unrolled steps.

In [ ]:
torch.manual_seed(SEED)
class ForecastLSTM(nn.Module):
 def __init__(self): super().__init__();self.rnn=nn.LSTM(1,20,batch_first=True);self.head=nn.Linear(20,1)
 def forward(self,x): z,(h,c)=self.rnn(x);return self.head(z[:,-1]),h
net=ForecastLSTM();opt=torch.optim.Adam(net.parameters(),lr=.01);losses=[]
for epoch in range(12):
 total=0
 for xb0,yb0 in loader: opt.zero_grad();p,h=net(xb0);loss=nn.functional.mse_loss(p,yb0);loss.backward();opt.step();total+=loss.item()*len(xb0)
 losses.append(total/cut)
with torch.no_grad(): scaled_pred,_=net(torch.tensor(Xb,dtype=torch.float32));pred=scaled_pred.numpy().ravel()*std+mean
print("Final hidden shape:",h.shape,"loss first/last:",round(losses[0],4),round(losses[-1],4))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
naive=X[cut:,-1];rmse=mean_squared_error(yb,pred)**.5;base=mean_squared_error(yb,naive)**.5;fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses);axes[0].set_title("Training MSE")
axes[1].plot(yb[:100],label="truth");axes[1].plot(pred[:100],label="LSTM");axes[1].plot(naive[:100],label="last value",alpha=.6);axes[1].legend();axes[1].set_title("Chronological holdout")
axes[2].scatter(yb,pred,s=10,alpha=.5);m=[yb.min(),yb.max()];axes[2].plot(m,m,"--",color="black");axes[2].set(title=f"RMSE {rmse:.3f} vs {base:.3f}",xlabel="truth",ylabel="prediction");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: A one-step LSTM trained on true history can accumulate its own errors when predictions are recursively fed back. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
window=values[-L:].copy();recursive=[]
for _ in range(120):
 inp=torch.tensor(((window-mean)/std)[None,:,None],dtype=torch.float32)
 with torch.no_grad(): nxt=net(inp)[0].item()*std+mean
 recursive.append(nxt);window=np.r_[window[1:],nxt]
truth_future=(1+.0005*np.arange(1400,1520))*np.sin(2*np.pi*np.arange(1400,1520)/40)+.35*np.sin(2*np.pi*np.arange(1400,1520)/9)
print(f"One-step RMSE={rmse:.3f}; 120-step recursive RMSE={mean_squared_error(truth_future,recursive)**.5:.3f}; final absolute error={abs(truth_future[-1]-recursive[-1]):.3f}")

**Use this when…** ordered data have moderate sequential dependence and an LSTM baseline is easy to validate.

        **Don't use this when…** long-range parallel context, huge datasets, or long recursive horizons favour attention or direct multi-horizon models.

        ## Try this

        1. Change window length from 8 to 80.
2. Predict 12 outputs directly instead of recursively.
3. Replace LSTM with a simple lag-linear model.